# 🛡️ FraudShield AI — Model Training (Google Colab)
### AI-Driven Fraud Detection in Financial Transactions
**DSE-B Mini Project · 2024–25**

---
**Instructions:**
1. Upload `creditcard.csv` (from Kaggle) using the upload cell below
2. Run all cells top to bottom (`Runtime → Run all`)
3. Download `fraud_model.pkl` and `scaler.pkl` from the Files panel
4. Place them in your local `model/` folder
5. Run `python app.py` on your computer to start the API

In [ ]:
# ─── CELL 1: Install libraries ───
!pip install imbalanced-learn --quiet
print('✅ Libraries ready')

In [ ]:
# ─── CELL 2: Upload dataset ───
from google.colab import files
print('📁 Upload your creditcard.csv file (from Kaggle)')
uploaded = files.upload()
print('✅ File uploaded:', list(uploaded.keys()))

In [ ]:
# ─── CELL 3: Imports ───
import os, json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
    roc_auc_score, f1_score, precision_score, recall_score, roc_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE

os.makedirs('model', exist_ok=True)
RANDOM_SEED = 42
print('✅ Imports done')

In [ ]:
# ─── CELL 4: Load & explore data ───
df = pd.read_csv('creditcard.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print('\nClass distribution:')
print(df['Class'].value_counts())
print(f'\nFraud rate: {df["Class"].mean()*100:.4f}%')
df.head()

In [ ]:
# ─── CELL 5: Data cleaning ───
print('Missing values:', df.isnull().sum().sum())
print('Duplicates:', df.duplicated().sum())
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)
print(f'Clean dataset: {len(df):,} rows')

# EDA plots
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fraud_count = df['Class'].sum()
legit_count = len(df) - fraud_count
axes[0].bar(['Legitimate', 'Fraudulent'], [legit_count, fraud_count],
            color=['#3266ad', '#c0392b'], alpha=0.85)
axes[0].set_title('Class Distribution')
df[df['Class']==0]['Amount'].hist(ax=axes[1], bins=60, alpha=0.6, label='Legit', color='#3266ad')
df[df['Class']==1]['Amount'].hist(ax=axes[1], bins=60, alpha=0.7, label='Fraud', color='#c0392b')
axes[1].set_title('Amount by Class')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# ─── CELL 6: Preprocessing ───
X = df.drop(columns=['Class'])
y = df['Class']
scaler = StandardScaler()
X[['Time','Amount']] = scaler.fit_transform(X[['Time','Amount']])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')
print(f'Train fraud: {y_train.sum()}  Test fraud: {y_test.sum()}')

In [ ]:
# ─── CELL 7: SMOTE ───
print('Before SMOTE:', dict(y_train.value_counts()))
smote = SMOTE(random_state=RANDOM_SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print('After  SMOTE:', dict(pd.Series(y_train_sm).value_counts()))

In [ ]:
# ─── CELL 8: Train models ───
print('Training Logistic Regression...')
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_SEED)
lr.fit(X_train_sm, y_train_sm)

print('Training Random Forest (this takes 2-3 minutes)...')
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                             random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)
print('✅ Both models trained!')

In [ ]:
# ─── CELL 9: Evaluate ───
for name, clf in [('Logistic Regression', lr), ('Random Forest', rf)]:
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:,1]
    print(f'\n═══ {name} ═══')
    print(f'Precision : {precision_score(y_test,y_pred):.4f}')
    print(f'Recall    : {recall_score(y_test,y_pred):.4f}')
    print(f'F1 Score  : {f1_score(y_test,y_pred):.4f}')
    print(f'AUC-ROC   : {roc_auc_score(y_test,y_prob):.4f}')
    print('\nConfusion Matrix:')
    print(confusion_matrix(y_test, y_pred))

# Confusion matrix plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, clf) in zip(axes, [('Logistic Reg', lr), ('Random Forest', rf)]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, clf.predict(X_test)),
                           display_labels=['Legit','Fraud']).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)
plt.tight_layout(); plt.show()

# ROC curves
fig, ax = plt.subplots(figsize=(7,5))
for name, clf, c, ls in [('LR',lr,'#73726c','--'),('RF',rf,'#3266ad','-')]:
    fpr,tpr,_ = roc_curve(y_test, clf.predict_proba(X_test)[:,1])
    auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:,1])
    ax.plot(fpr, tpr, color=c, lw=2, ls=ls, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--',alpha=0.3,label='Random')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curves'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ─── CELL 10: Save model and metrics ───
joblib.dump(rf, 'model/fraud_model.pkl')
joblib.dump(scaler, 'model/scaler.pkl')

rf_pred = rf.predict(X_test)
lr_pred = lr.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:,1]
lr_prob = lr.predict_proba(X_test)[:,1]
cm_rf = confusion_matrix(y_test, rf_pred).tolist()
cm_lr = confusion_matrix(y_test, lr_pred).tolist()

metrics = {
  'random_forest': {
    'precision': round(precision_score(y_test,rf_pred),4),
    'recall':    round(recall_score(y_test,rf_pred),4),
    'f1_score':  round(f1_score(y_test,rf_pred),4),
    'auc_roc':   round(roc_auc_score(y_test,rf_prob),4),
    'confusion_matrix': cm_rf
  },
  'logistic_regression': {
    'precision': round(precision_score(y_test,lr_pred),4),
    'recall':    round(recall_score(y_test,lr_pred),4),
    'f1_score':  round(f1_score(y_test,lr_pred),4),
    'auc_roc':   round(roc_auc_score(y_test,lr_prob),4),
    'confusion_matrix': cm_lr
  },
  'dataset_info': {
    'total_transactions': int(len(df)),
    'fraud_count':        int(fraud_count),
    'legitimate_count':   int(legit_count),
    'fraud_percentage':   round(df['Class'].mean()*100, 4),
    'features':           X.columns.tolist()
  }
}
with open('model/metrics.json','w') as f:
    json.dump(metrics, f, indent=2)

print('✅ Saved: model/fraud_model.pkl')
print('✅ Saved: model/scaler.pkl')
print('✅ Saved: model/metrics.json')

In [ ]:
# ─── CELL 11: Download model files to your computer ───
from google.colab import files
print('Downloading model files...')
files.download('model/fraud_model.pkl')
files.download('model/scaler.pkl')
files.download('model/metrics.json')
print('✅ Download started! Place these files in your local model/ folder.')